In [3]:
import numpy as np
from scipy.stats import laplace, norm

def run_simulation(dist_type='gaussian', n=100, n_sims=10000):
    p = 0.5
    z_alpha = 1.96
    
    margin = z_alpha * np.sqrt(p * (1 - p)) / np.sqrt(n)
    pn, qn = p - margin, p + margin
    
    idx_low = int(np.ceil(n * pn)) - 1
    idx_high = int(np.ceil(n * qn)) - 1

    mean_hits = 0
    median_hits = 0
    
    for _ in range(n_sims):
        if dist_type == 'gaussian':
            sample = np.random.normal(0, 1, n)
        else:
            sample = np.random.laplace(0, 1, n)
            
        # Mean-based Confidence Interval
        x_bar = np.mean(sample)
        s_n = np.std(sample, ddof=1)
        ci_mean_low = x_bar - z_alpha * (s_n / np.sqrt(n))
        ci_mean_high = x_bar + z_alpha * (s_n / np.sqrt(n))
        
        if ci_mean_low <= 0 <= ci_mean_high:
            mean_hits += 1
            
        # Median-based Confidence Interval (Order Statistics)
        sorted_sample = np.sort(sample)
        ci_med_low = sorted_sample[idx_low]
        ci_med_high = sorted_sample[idx_high]
        
        if ci_med_low <= 0 <= ci_med_high:
            median_hits += 1
            
    return mean_hits / n_sims, median_hits / n_sims

gauss_results = run_simulation('gaussian')
laplace_results = run_simulation('laplace')

print(f"Gaussian Data | Mean CI Coverage: {gauss_results[0]:.4f} | Median CI Coverage: {gauss_results[1]:.4f}")
print(f"Laplace Data  | Mean CI Coverage: {laplace_results[0]:.4f} | Median CI Coverage: {laplace_results[1]:.4f}")

Gaussian Data | Mean CI Coverage: 0.9469 | Median CI Coverage: 0.9382
Laplace Data  | Mean CI Coverage: 0.9484 | Median CI Coverage: 0.9428
